# Vanilla RAG baseline on the competition training data

Builds a hybrid retrieval pipeline (BM25 + MiniLM dense) over a Wikipedia corpus seeded from proper-noun mentions in the training prompts, scores each option for a held-out probe of train rows, and logs MAP@3 to W&B.

Baseline to beat: 0.20 (label prior). Interpretation guide below the final cell.

In [1]:
import os
import sys
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# make the src package importable when the notebook lives in notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.normalize import normalize_stem
from src.rag.corpus import fetch_wikipedia_passages, save_corpus_to_json, load_corpus_from_json
from src.rag.chunk import chunk_passage
from src.rag.index import HybridIndex
from src.rag.score import score_options_with_rag, score_to_probs
from src.config import DATA_DIR, ARTIFACTS_DIR

import wandb

## 1. Load competition data

In [2]:
train = pd.read_csv(os.path.join(r"C:\Users\Krishna\Downloads\dl-genai-t22026-23f2001849\smart-mcq-solver-challenge", "train.csv"))
train["core_q"] = train["prompt"].apply(normalize_stem)
print(f"Train rows: {len(train)}")
print(f"Unique core_q: {train['core_q'].nunique()}")
train.head(3)

Train rows: 2000
Unique core_q: 252


,id,prompt,A,B,C,D,E,answer,core_q
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,What is Martin Heidegger's view on the relatio...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,What is accelerator-based light-ion fusion?
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,What is the term used in astrophysics to descr...


## 2. Seed retrieval topics from the training prompts

Extract multi-word capitalized proper nouns from every prompt and count occurrences. The top-N most frequent become Wikipedia query seeds. This grounds the corpus in the actual content of the competition rather than a generic STEM list.

In [3]:
_NOUN_RE = re.compile(r"\b[A-Z][a-z]{3,}(?:\s+[A-Z][a-z]+)*\b")
_STOP = {"Which", "What", "Who", "The", "This", "That", "How", "Where", "When",
         "Pick", "Select", "Determine", "Identify", "Choose"}

def extract_topics(prompts, top_n=100):
    counts = {}
    for p in prompts:
        for m in _NOUN_RE.findall(str(p)):
            if m in _STOP:
                continue
            counts[m] = counts.get(m, 0) + 1
    return sorted(counts.items(), key=lambda x: -x[1])[:top_n]

topic_counts = extract_topics(train["prompt"].tolist(), top_n=100)
seed_topics = [t for t, _ in topic_counts]
print(f"Extracted {len(seed_topics)} topics")
print("Top 30:", seed_topics[:30])

Extracted 90 topics
Top 30: ['Earth', 'Einstein', 'Newton', 'Modified Newtonian Dynamics', 'Lorentz', 'Navier', 'Stokes', 'Isaac Newton', 'Gravity Probe', 'Pierre', 'Fermat', 'Doppler', 'Minkowski', 'Maxwell', 'Demon', 'Liouville', 'Relativity', 'Kutta', 'Peierls', 'Optical Signal', 'Noise Ratio', 'Stefan', 'Boltzmann', 'Environmental Science Center', 'Qatar University', 'Home', 'Evans', 'Hamiltonians', 'Ozma Problem', 'Landau']


## 3. Fetch Wikipedia corpus (cached)

Uses the Wikipedia REST summary endpoint. Cached to `artifacts/rag_corpus.json` so subsequent runs skip the network round-trip. First run takes ~1-2 minutes for 100 topics.

In [4]:
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
corpus_path = os.path.join(ARTIFACTS_DIR, "rag_corpus.json")

if os.path.exists(corpus_path):
    passages = load_corpus_from_json(corpus_path)
    print(f"Loaded {len(passages)} cached passages")
else:
    passages = fetch_wikipedia_passages(topics=seed_topics, cache_path=corpus_path)
    print(f"Fetched {len(passages)} passages, cached to {corpus_path}")

if passages:
    print("\nExample passage:")
    print(f"  Title: {passages[0]['title']}")
    print(f"  Text: {passages[0]['text'][:200]}...")

Loaded 27 cached passages

Example passage:
  Title: Earth
  Text: Earth is the third planet from the Sun and the only astronomical object known to harbor life. This is made possible by Earth being an ocean world, the only one in the Solar System sustaining liquid su...


## 4. Chunk the corpus

Sentence-boundary chunking with a 120-word cap and 1-sentence overlap. Overlap prevents context loss at chunk boundaries.

In [5]:
chunks = []
for p in passages:
    for i, ch in enumerate(chunk_passage(p["text"], max_words=120, overlap_sentences=1)):
        chunks.append({"id": f"{p['id']}_{i}", "text": ch, "source": p["id"]})

print(f"Total chunks: {len(chunks)}")
print(f"Mean chunk length: {np.mean([len(c['text'].split()) for c in chunks]):.1f} words")
print(f"Max chunk length:  {max(len(c['text'].split()) for c in chunks)} words")

Total chunks: 29
Mean chunk length: 54.6 words
Max chunk length:  113 words


## 5. Build the hybrid index

BM25 over lowercased alphanumeric tokens plus MiniLM dense embeddings. First run loads the sentence-transformer weights (~90 MB) and encodes all chunks — expect 1-2 minutes on CPU.

In [6]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
index = HybridIndex(chunks, embedder=embedder)
print("Index built.")
print(f"  BM25 vocabulary items indexed: {len(chunks)}")
print(f"  Dense matrix shape: {index.dense_matrix.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built.
  BM25 vocabulary items indexed: 29
  Dense matrix shape: (29, 384)


## 6. Score a held-out probe from train

Uses the last 200 train rows as the probe. Keeps the first 1800 rows unseen for downstream training arms so cross-validation isn't polluted by RAG-probe overlap. For each row, compute a retrieval-augmented score per option, rank the five options, take the top-3.

In [8]:
import os
os.environ["WANDB_API_KEY"] = "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"
wandb.login(key=os.environ["WANDB_API_KEY"])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Krishna\_netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [9]:
probe = train.tail(200).reset_index(drop=True)

def top3_from_scores(scores):
    """Sort letters by descending score, return top-3 as space-joined string."""
    order = sorted(scores.items(), key=lambda x: -x[1])
    return " ".join(L for L, _ in order[:3])

wandb.init(
    project="23f2001849-t22026",
    name="rag_vanilla_baseline",
    tags=["rag", "vanilla", "probe200"],
    config={
        "embedder": "all-MiniLM-L6-v2",
        "chunker": "sentence-boundary",
        "max_words": 120,
        "overlap_sentences": 1,
        "top_k": 5,
        "alpha": 0.5,
        "probe_size": len(probe),
        "corpus_size": len(chunks),
        "topic_seed_count": len(seed_topics),
    },
)

preds = []
for idx, row in probe.iterrows():
    options = {L: str(row[L]) for L in "ABCDE"}
    scores = score_options_with_rag(
        prompt=row["prompt"],
        options=options,
        index=index,
        reranker=None,
        top_k=5,
        alpha=0.5,
    )
    preds.append(top3_from_scores(scores))
    if (idx + 1) % 25 == 0:
        print(f"  scored {idx + 1} / {len(probe)}")

probe["pred_top3"] = preds
print("\nDone scoring.")

  scored 25 / 200
  scored 50 / 200
  scored 75 / 200
  scored 100 / 200
  scored 125 / 200
  scored 150 / 200
  scored 175 / 200
  scored 200 / 200

Done scoring.


## 7. MAP@3 evaluation

Standard MAP@3 formula: reward 1.0 for rank-1 correct, 0.5 for rank-2, 0.333 for rank-3, 0 otherwise. Compare against a label-prior baseline ("always predict B C A", the three most frequent letters in train).

In [10]:
def map_at_3(y_true, y_pred_top3):
    total = 0.0
    for true, pred in zip(y_true, y_pred_top3):
        pred_letters = pred.split()
        for rank, letter in enumerate(pred_letters[:3], start=1):
            if letter == true:
                total += 1.0 / rank
                break
    return total / len(y_true)

rag_score = map_at_3(probe["answer"].tolist(), probe["pred_top3"].tolist())
prior_score = map_at_3(probe["answer"].tolist(), ["B C A"] * len(probe))

print(f"Vanilla RAG MAP@3:       {rag_score:.4f}")
print(f"Label-prior MAP@3:       {prior_score:.4f}")
print(f"Lift over prior:         {rag_score - prior_score:+.4f}")

wandb.log({
    "map_at_3": rag_score,
    "label_prior_map_at_3": prior_score,
    "lift_over_prior": rag_score - prior_score,
})

Vanilla RAG MAP@3:       0.3500
Label-prior MAP@3:       0.4308
Lift over prior:         -0.0808


## 8. Per-tier decomposition

Break MAP@3 down by whether the probe row was a T1 lookup hit or not, to isolate where RAG contributes signal versus where the lookup already dominates.

In [11]:
from src.lookup import build_lookup_tables, lookup

# build lookup from the 1800 rows outside the probe
lookup_train = train.head(1800).copy()
tables = build_lookup_tables(lookup_train)

tier_col = []
for _, r in probe.iterrows():
    _, tier = lookup(r, tables)
    tier_col.append(tier if tier else "MISS")
probe["tier"] = tier_col

print("Probe row counts by tier:")
print(probe["tier"].value_counts())

print("\nRAG MAP@3 by tier:")
for tier in probe["tier"].unique():
    sub = probe[probe["tier"] == tier]
    if len(sub) > 0:
        s = map_at_3(sub["answer"].tolist(), sub["pred_top3"].tolist())
        print(f"  {tier}: {s:.4f}  (n={len(sub)})")
        wandb.log({f"map_at_3_{tier}": s, f"n_{tier}": len(sub)})

Probe row counts by tier:
tier
T1      152
T3       41
MISS      4
T2        3
Name: count, dtype: int64

RAG MAP@3 by tier:
  T1: 0.3596  (n=152)
  T3: 0.3455  (n=41)
  T2: 0.0000  (n=3)
  MISS: 0.2917  (n=4)


## 9. Persist predictions and finish

In [12]:
out_path = os.path.join(ARTIFACTS_DIR, "rag_vanilla_probe_predictions.csv")
probe[["id", "prompt", "answer", "pred_top3", "tier"]].to_csv(out_path, index=False)
print(f"Saved predictions to {out_path}")

wandb.finish()

Saved predictions to ./outputs\artifacts\rag_vanilla_probe_predictions.csv


label_prior_map_at_3,▁
lift_over_prior,▁
map_at_3,▁
map_at_3_MISS,▁
map_at_3_T1,▁
map_at_3_T2,▁
map_at_3_T3,▁
n_MISS,▁
n_T1,▁
n_T2,▁
+1,...


## Interpretation

- **RAG MAP@3 above 0.35**: retrieval has real signal. Worth adding a reranker (already built at `src/rag/rerank.py`), swapping embedder, or feeding scores into the ensemble grid as a 6th arm.
- **RAG MAP@3 between 0.20 and 0.30**: marginal. Note it as an ablation in the report, don't invest more time.
- **RAG MAP@3 at or below 0.20**: no signal above prior. The retrieval task and the MCQA task don't align on this corpus. Note as a failed experiment in the report and move on.
- **Per-tier breakdown**: T1 hits are already resolved by lookup at 0.74 floor. RAG's real leverage is on MISS rows and paraphrase-flipped T1 rows. If per-tier MAP@3 on MISS is strong, RAG is worth ensembling.